In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import sys
from pathlib import Path

ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT))

from src.config.paths import *

ANALYSIS_DIR = CSV_DIR / "data_analysis"
IMAGE_DIR = IMAGE_DIR / "data_analysis"
PLOT_DIR = PLOT_DIR / "data_analysis"

In [ ]:
image_stats = pd.read_csv(ANALYSIS_DIR/"image_properties.csv")
box_sizes = pd.read_csv(ANALYSIS_DIR/"box_sizes.csv")

In [ ]:
combined = box_sizes.merge(
    image_stats,
    on="path",
    how="left",
    validate="many_to_one"
)

combined

In [ ]:
print("Correlation between bounding box area and image area")

print("Pearson:", combined["area_x"].corr(combined["area_y"], method="pearson"))
print("Spearman:", combined["area_x"].corr(combined["area_y"], method="spearman"))

In [ ]:
combined["box_count"] = combined.groupby("path")["path"].transform("size")

print("Correlation between box count and its area")

print("Pearson:", combined["box_count"].corr(combined["area_x"], method="pearson"))
print("Spearman:", combined["box_count"].corr(combined["area_x"], method="spearman"))

In [ ]:
# Just wondering does box count artificially increase/decrease the correlation in above checks 
image_boxes = combined.groupby("path").agg(
    mean_box_area=("area_x", "mean"),
    median_box_area=("area_x", "median"),
    max_box_area=("area_x", "max"),
    min_box_area=("area_x", "min"),
    box_count=("area_x", "size"),
    image_area=("area_y", "first")
)

image_boxes

In [ ]:
pears_corr = image_boxes.corr(method="pearson")
spear_corr = image_boxes.corr(method="spearman")

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

for ax, corr, title in zip(
    axes,
    [pears_corr, spear_corr],
    ["Pearson Correlation", "Spearman Correlation"]
):

    # Keep only correlations needed for image properties
    corr = corr.loc[
        ["mean_box_area", "median_box_area", "max_box_area",
            "min_box_area", "box_count"],
        ["box_count", "image_area"]
    ]

    sns.heatmap(
        corr,
        ax=ax,
        cbar=False,
        annot=True,
        vmin=-1,
        vmax=1,
        center=0,
        cmap="viridis",
        fmt=".2f",
        square=True,
        annot_kws={"size": 10}
    )

    ax.set_title(title)

plt.tight_layout()
plt.savefig(PLOT_DIR / "correlations.svg", bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 10))

for i, (category, name) in enumerate([(0, "Smoke"), (1, "Fire")]):
    subset = combined[combined["category_x"] == category]

    image_boxes = subset.groupby("path").agg(
        mean_box_area=("area_x", "mean"),
        median_box_area=("area_x", "median"),
        max_box_area=("area_x", "max"),
        min_box_area=("area_x", "min"),
        box_count=("area_x", "size"),
        image_area=("area_y", "first")
    )

    for j, method in enumerate(["pearson", "spearman"]):
        corr = image_boxes.corr(method=method)

        # Keep only correlations needed for image properties
        corr = corr.loc[
            ["mean_box_area", "median_box_area", "max_box_area",
             "min_box_area", "box_count"],
            ["box_count", "image_area"]
        ]

        sns.heatmap(
            corr,
            ax=axes[i, j],
            cbar=False,
            annot=True,
            vmin=-1,
            vmax=1,
            center=0,
            cmap="viridis",
            fmt=".2f",
            annot_kws={"size": 10}
        )

        axes[i, j].set_title(f"{name} - {method.capitalize()}")

plt.tight_layout()
plt.show()